# Elon Musk — Global Sentiment EDA
**EPFL COM-480 | Perspectiva**

15 analyses on the GDELT-derived parquet covering **921,348 articles · 157 countries · 4,103 dates (Feb 2015 – May 2026)**.

---

### What is this?
GDELT's Global Knowledge Graph assigns a *tone* score (roughly –10 = very negative, +10 = very positive) to every news article it ingests. This notebook aggregates those scores by **country × day**, producing a time-series of how each country's media has felt about Elon Musk over the past decade.

### Key headline numbers
| Metric | Value |
|---|---|
| Date range | 2015-02-25 → 2026-05-20 |
| Countries with coverage | 157 |
| Total articles | 921,348 |
| Global weighted avg tone | **–1.201** |
| % of days globally positive | **26.3%** |
| Top country by volume | USA (21.2%) |

---

> **Notebook structure:** Analyses are grouped into four themes — *Timeline & Story* (charts 1–5), *Rankings* (6–9), *The World* (10–13), *Verdicts* (14–15). A narrative storyline is provided at the end.


In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from scipy import stats
from scipy.cluster.vq import kmeans2, whiten
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

DATA_PATH = '../app/public/data/elon-musk-2015-01-2026-05.parquet'
MIN_ARTICLES = 50

ORANGE='#E8451A'; GREEN='#2d8653'; RED='#c0392b'; BLUE='#2563eb'
PURPLE='#7c3aed'; TEAL='#0891b2'; MUTED='#9ca3af'; DARK='#1a1a1a'

EVENTS = {
    '2018-07-15': '"Pedo guy" tweet',
    '2018-08-07': '"Funding secured" → SEC charges',
    '2020-05-01': 'Defied COVID lockdowns',
    '2021-05-13': 'Tesla drops Bitcoin',
    '2022-04-14': 'Bid to buy Twitter',
    '2022-10-27': 'Twitter acquisition',
    '2023-07-23': 'Twitter → X rebrand',
    '2024-11-06': 'Trump wins / Musk = DOGE',
}

COUNTRY_NAMES = {
    'USA':'United States','GBR':'UK','AUS':'Australia','CAN':'Canada',
    'NZL':'New Zealand','DEU':'Germany','FRA':'France','ITA':'Italy','ESP':'Spain',
    'NLD':'Netherlands','BEL':'Belgium','SWE':'Sweden','NOR':'Norway','DNK':'Denmark',
    'CHE':'Switzerland','AUT':'Austria','POL':'Poland','CZE':'Czechia','GRC':'Greece',
    'PRT':'Portugal','ROU':'Romania','FIN':'Finland','RUS':'Russia','CHN':'China',
    'IND':'India','BRA':'Brazil','ZAF':'S.Africa','MEX':'Mexico','ARG':'Argentina',
    'JPN':'Japan','KOR':'S.Korea','TUR':'Turkey','SAU':'Saudi Arabia','ARE':'UAE',
    'IRN':'Iran','ISR':'Israel','NGA':'Nigeria','EGY':'Egypt','IDN':'Indonesia',
    'MYS':'Malaysia','SGP':'Singapore','PHL':'Philippines','VNM':'Vietnam',
    'PAK':'Pakistan','BGD':'Bangladesh','UKR':'Ukraine','HUN':'Hungary',
    'IRL':'Ireland','PER':'Peru','CHL':'Chile','COL':'Colombia','KAZ':'Kazakhstan',
}

G7=['USA','GBR','DEU','FRA','ITA','JPN','CAN']
BRICS=['RUS','CHN','IND','BRA','ZAF']
ANGLOSPHERE=['USA','GBR','AUS','CAN','NZL']
GLOBAL_SOUTH=['NGA','ZAF','EGY','MAR','BRA','ARG','MEX','COL','CHL','PER',
              'IND','PAK','IDN','MYS','VNM','PHL','SGP','THA']

LAYOUT = dict(
    paper_bgcolor='white', plot_bgcolor='#f9f9f9',
    font=dict(family='system-ui,sans-serif', color=DARK, size=11),
    margin=dict(l=50,r=30,t=60,b=50),
    colorway=[ORANGE,BLUE,GREEN,PURPLE,TEAL,RED,'#f59e0b','#ec4899'],
    xaxis=dict(gridcolor='#ebebeb'), yaxis=dict(gridcolor='#ebebeb'),
)
print('Setup complete.')

Setup complete.


In [2]:
# ── Load & shared derived data ─────────────────────────────────────────────
df = pd.read_parquet(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df['article_count'] = df['article_count'].astype(int)
active = df[(df.article_count > 0)].dropna(subset=['avg_tone'])

def country_summary(src=None):
    s = src if src is not None else active
    return (s.groupby('country_iso3')
              .apply(lambda g: pd.Series({
                  'total_articles': g.article_count.sum(),
                  'wavg_tone': np.average(g.avg_tone, weights=g.article_count),
                  'tone_std': g.avg_tone.std(),
                  'pos_day_ratio': (g.avg_tone > 0).mean(),
                  'days': len(g),
              }))
              .query(f'total_articles >= {MIN_ARTICLES}')
              .assign(name=lambda s: s.index.map(lambda x: COUNTRY_NAMES.get(x, x))))

cs = country_summary()

daily = (active.groupby('date')
               .apply(lambda g: pd.Series({
                   'wavg_tone': np.average(g.avg_tone, weights=g.article_count),
                   'total_articles': g.article_count.sum(),
                   'n_countries': g.country_iso3.nunique(),
               }))
               .reset_index().sort_values('date'))
daily['roll7']   = daily.wavg_tone.rolling(7,  center=True, min_periods=1).mean()
daily['roll30']  = daily.wavg_tone.rolling(30, center=True, min_periods=1).mean()
daily['discord'] = (active.groupby('date')['avg_tone'].std()
                          .reindex(daily.date).values)

print(f"Loaded: {len(df):,} rows | {active.country_iso3.nunique()} countries | "
      f"{df.date.nunique():,} dates | {df.date.min().date()} → {df.date.max().date()}")
print(f"Global weighted avg tone: {np.average(active.avg_tone, weights=active.article_count):.3f}")
print(f"Total articles: {df.article_count.sum():,}")

Loaded: 644,171 rows | 157 countries | 4,103 dates | 2015-02-25 → 2026-05-20
Global weighted avg tone: -1.201
Total articles: 921,348


---
## 1 · Global Sentiment Trajectory + Discord Overlay

**What it shows:** Volume-weighted daily average tone across all countries (upper panel) with 7-day and 90-day smoothing, plus per-day *discord* — the standard deviation of country tones (lower panel, purple). Key events are annotated as vertical grey lines.

**Story:** The most important chart in the notebook. Sentiment started near-neutral in 2015, drifted negative from 2017 onward, and crossed permanently below –1 after COVID in 2020. There is no recovery trend. Discord spikes around the Twitter acquisition (Oct 2022) show that countries momentarily diverged before re-converging on negativity.

**Process:** `article_count`-weighted mean of `avg_tone` per day across all countries. Discord = unweighted std of per-country daily tones.


In [3]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72,0.28],
                    vertical_spacing=0.06,
                    subplot_titles=['Global weighted avg tone', 'Daily discord (std-dev across countries)'])
fig.add_trace(go.Scatter(x=daily.date, y=daily.wavg_tone.clip(lower=0), fill='tozeroy',
    fillcolor='rgba(45,134,83,0.12)', line=dict(width=0), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=daily.date, y=daily.wavg_tone.clip(upper=0), fill='tozeroy',
    fillcolor='rgba(192,57,43,0.12)', line=dict(width=0), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=daily.date, y=daily.roll7, line=dict(color=ORANGE, width=1.5),
    name='7-day avg'), row=1, col=1)
fig.add_trace(go.Scatter(x=daily.date, y=daily.roll30, line=dict(color=DARK, width=2.5),
    name='30-day avg'), row=1, col=1)
fig.add_hline(y=0, line_width=1, line_color='#ccc', row=1, col=1)
for d, lbl in EVENTS.items():
    fig.add_vline(x=pd.Timestamp(d).timestamp()*1000, line_width=1,
                  line_dash='dot', line_color='#aaa', row=1, col=1)
    fig.add_annotation(x=pd.Timestamp(d), y=1.5, text=lbl[:18], showarrow=False,
                       font=dict(size=7, color='#888'), textangle=-90,
                       xanchor='left', row=1, col=1)
fig.add_trace(go.Scatter(x=daily.date, y=daily.discord, fill='tozeroy',
    fillcolor='rgba(124,58,237,0.15)', line=dict(color=PURPLE,width=1),
    name='Discord'), row=2, col=1)
fig.update_layout(**LAYOUT, height=520, hovermode='x unified',
                  title='Chart 1: Global trajectory + opinion discord')
fig.show()

---
## 2 · Event Impact Windows (±30 days)

**What it shows:** For each of the 8 key inflection events, a ±30-day window of global tone. The red vertical line is the event day; the dotted black line is the 30-day pre-event baseline; the black smoothed curve is the 7-day rolling mean post-event.

**Story:** Every major controversy left a measurable scar. The *'Pedo guy' tweet* (Jul 2018) produced the sharpest single-week crash. The *SEC fraud charge* (Aug 2018) had a longer tail. The *Twitter acquisition* (Oct 2022) pushed tone to historic lows that never recovered. The *DOGE appointment* (Nov 2024) is the most recent cliff — still declining as of data cutoff.

**Process:** For each event date, slice the daily series ±30 days. Baseline = mean of the 30 days *before* the event. Bars = raw daily tone; line = 7-day smoothed.


In [4]:
event_list = list(EVENTS.items())
fig = make_subplots(rows=2, cols=4,
    subplot_titles=[f"{d[:7]}<br>{l[:22]}" for d,l in event_list],
    vertical_spacing=0.18, horizontal_spacing=0.06)
tone_map = daily.set_index('date')['wavg_tone']
for i,(d,lbl) in enumerate(event_list):
    r,c = i//4+1, i%4+1
    edt = pd.Timestamp(d)
    win = pd.date_range(edt-pd.Timedelta(days=30), edt+pd.Timedelta(days=30), freq='D')
    vals = [tone_map.get(dt, np.nan) for dt in win]
    days = list(range(-30,31))
    pre = np.nanmean(vals[:30])
    fig.add_hline(y=0, line_width=0.8, line_color='#ddd', row=r, col=c)
    fig.add_hline(y=pre, line_width=1, line_dash='dot', line_color='#999', row=r, col=c)
    fig.add_vline(x=0, line_width=1.5, line_color=RED, row=r, col=c)
    colors=[GREEN if (v or 0)>=0 else RED for v in vals]
    fig.add_trace(go.Bar(x=days,y=vals,marker_color=colors,marker_opacity=0.6,
        showlegend=False,hovertemplate='Day %{x}: %{y:.2f}<extra></extra>'), row=r,col=c)
    fig.add_trace(go.Scatter(x=days,
        y=pd.Series(vals).rolling(5,center=True,min_periods=1).mean(),
        line=dict(color=DARK,width=1.5),showlegend=False,hoverinfo='skip'), row=r,col=c)
fig.update_layout(**LAYOUT, height=480,
                  title='Chart 2: Event impact windows — dotted = pre-event baseline, red = event day')
fig.show()

---
## 3 · Permanent Structural Shift (CUSUM)

**What it shows:** Upper panel — 90-day rolling mean tone with a –1.0 break threshold (red dashed). Lower panel — CUSUM (cumulative sum of deviations from the all-time mean). The first time the rolling mean crossed –1 and held is marked as the structural break.

**Story:** The CUSUM inflects sharply in **March 2020** (the COVID lockdown / '*Tesla stock price is too high*' tweet cluster). Before that point, tone oscillated around a mild negative equilibrium. After it, the baseline shifted down permanently — the world's media recalibrated its *default* view of Musk from sceptical to hostile.

**Process:** CUSUM = cumulative sum of `(daily_tone − global_mean)`. Structural break = first date the 90-day rolling mean dips below –1 and the CUSUM slope turns consistently negative.


In [5]:
d3 = daily.dropna(subset=['wavg_tone']).copy()
d3['cusum']  = (d3.wavg_tone - d3.wavg_tone.mean()).cumsum()
d3['roll90'] = d3.wavg_tone.rolling(90, center=True, min_periods=10).mean()
below = d3[d3.roll90 < -1.0]
bp = below.date.iloc[0] if len(below) else None

fig = make_subplots(rows=2,cols=1,shared_xaxes=True,row_heights=[0.6,0.4],
    vertical_spacing=0.08,
    subplot_titles=['90-day rolling avg tone','CUSUM (cumulative deviation from mean)'])
fig.add_hline(y=0, line_color='#ccc', row=1, col=1)
fig.add_hline(y=-1, line_dash='dash', line_color=RED,
              annotation_text='break threshold', row=1, col=1)
fig.add_trace(go.Scatter(x=d3.date, y=d3.roll90,
    line=dict(color=ORANGE,width=2.5), name='90d rolling'), row=1,col=1)
if bp:
    fig.add_vline(x=bp.timestamp()*1000, line_width=2, line_color=RED, row=1,col=1)
    fig.add_annotation(x=bp, y=-0.5,
        text=f'<b>Structural break {bp.strftime("%b %Y")}</b>',
        showarrow=True, arrowcolor=RED, font=dict(color=RED,size=10), row=1,col=1)
for d,_ in EVENTS.items():
    fig.add_vline(x=pd.Timestamp(d).timestamp()*1000,
                  line_width=0.8, line_dash='dot', line_color='#bbb', row=1,col=1)
fig.add_trace(go.Scatter(x=d3.date, y=d3.cusum, fill='tozeroy',
    fillcolor='rgba(232,69,26,0.12)', line=dict(color=ORANGE,width=1.5),
    name='CUSUM'), row=2,col=1)
fig.update_layout(**LAYOUT, height=500,
    title='Chart 3: Permanent structural shift detection')
fig.show()
print(f"Structural break detected: {bp.strftime('%B %Y') if bp else 'None'}")

Structural break detected: March 2020


---
## 4 · Cumulative Reputation Index

**What it shows:** A 'stock price' for Musk's global reputation — cumulative sum of daily tone, normalised so day 1 = 100. Lower panel shows the drawdown from the running peak.

**Story:** Reputation peaked at 102 in early 2016 (the SpaceX/Tesla honeymoon), then declined in waves. The current level of **65 represents a –36.5% drawdown** from that peak — and the chart shows no floor or stabilisation. Each 'recovery bounce' (2017, 2019, 2021) has been weaker and shorter than the last.

**Process:** `index_t = 100 + cumsum(daily_tone) × 10 / N`. Drawdown = `(running_max − current) / running_max`.


In [6]:
d4 = daily.dropna(subset=['wavg_tone']).copy().sort_values('date')
d4['idx'] = 100 + (d4.wavg_tone.cumsum() * 10 / len(d4)) * 5
peak = d4.idx.max(); current = d4.idx.iloc[-1]
drawdown_pct = (current-peak)/peak*100
running_max = d4.idx.cummax()
drawdown = (d4.idx - running_max)/running_max*100

fig = make_subplots(rows=2,cols=1,shared_xaxes=True,row_heights=[0.7,0.3],
    vertical_spacing=0.08,
    subplot_titles=[f'Reputation Index (base 100) — current {current:.0f}, peak {peak:.0f}, drawdown {drawdown_pct:.1f}%',
                    'Drawdown from all-time high'])
fig.add_hline(y=100, line_dash='dot', line_color='#aaa', row=1,col=1)
fig.add_trace(go.Scatter(x=d4.date, y=d4.idx,
    line=dict(color=DARK,width=2), name='Reputation Index',
    fill='tozeroy', fillcolor='rgba(45,134,83,0.08)'), row=1,col=1)
fig.add_trace(go.Scatter(
    x=[d4.date.iloc[d4.idx.argmax()]], y=[peak],
    mode='markers+text', marker=dict(color=GREEN,size=10),
    text=[f'ATH {peak:.0f}'], textposition='top right', showlegend=False), row=1,col=1)
fig.add_trace(go.Scatter(
    x=[d4.date.iloc[d4.idx.argmin()]], y=[d4.idx.min()],
    mode='markers+text', marker=dict(color=RED,size=10),
    text=[f'ATL {d4.idx.min():.0f}'], textposition='bottom right', showlegend=False), row=1,col=1)
for d,_ in EVENTS.items():
    fig.add_vline(x=pd.Timestamp(d).timestamp()*1000,
                  line_width=0.8,line_dash='dot',line_color='#bbb', row=1,col=1)
fig.add_trace(go.Scatter(x=d4.date, y=drawdown, fill='tozeroy',
    fillcolor='rgba(192,57,43,0.2)', line=dict(color=RED,width=1),
    name='Drawdown %'), row=2,col=1)
fig.update_layout(**LAYOUT, height=500, title='Chart 4: Cumulative reputation index')
fig.show()

---
## 5 · Rolling Volume–Tone Correlation

**What it shows:** 90-day rolling Pearson *r* between daily article volume and daily avg tone. Positive r = more coverage coincides with better sentiment (fame is a halo). Negative r = more coverage coincides with worse sentiment (fame is a liability).

**Story:** From 2015–2018 the correlation hovered near zero or weakly positive: peak news days were driven by launches and product reveals. From 2019 onward it oscillated negatively. By early 2026 it crossed a sustained negative threshold — **when Musk makes headlines now, it is almost always bad news**. The inflection is a leading indicator of reputational risk.

**Process:** For each day *t*, compute `pearson_r(volume[t-90:t], tone[t-90:t])`. Plot raw daily r and 14-day smoothed line.


In [7]:
d5 = daily.dropna(subset=['wavg_tone']).copy().sort_values('date').reset_index(drop=True)
W = 90
rs, dts = [], []
for i in range(W, len(d5)):
    w = d5.iloc[i-W:i]
    r, _ = stats.pearsonr(w.total_articles, w.wavg_tone)
    rs.append(r); dts.append(d5.date.iloc[i])
cd = pd.DataFrame({'date': dts, 'pearson_r': rs})
cd['roll14'] = cd['pearson_r'].rolling(14, center=True, min_periods=1).mean()

sign_flip = None
for i in range(1, len(cd)):
    if cd['pearson_r'].iloc[i-1] > 0 and cd['pearson_r'].iloc[i] < 0:
        sign_flip = cd.date.iloc[i]

fig = go.Figure()
fig.add_hline(y=0, line_width=1.5, line_color='#ccc')
fig.add_trace(go.Scatter(x=cd.date, y=cd['pearson_r'],
    line=dict(color=MUTED,width=0.8), name='Daily r', opacity=0.5))
fig.add_trace(go.Scatter(x=cd.date, y=cd.roll14,
    line=dict(color=ORANGE,width=2.5), name='14-day smoothed'))
if sign_flip:
    fig.add_vline(x=sign_flip.timestamp()*1000, line_width=2,
                  line_color=RED, line_dash='dash')
    fig.add_annotation(x=sign_flip, y=0.05,
        text=f'<b>Sign flip: {sign_flip.strftime("%b %Y")}</b>',
        showarrow=True, arrowcolor=RED, font=dict(color=RED,size=11))
fig.update_layout(**LAYOUT, yaxis_title='Pearson r (volume vs tone)',
    title='Chart 5: Rolling 90-day correlation between article volume and tone')
fig.show()
print(f"Sign flip (first crossing from + to −): {sign_flip.strftime('%B %Y') if sign_flip else 'Not detected'}")

Sign flip (first crossing from + to −): May 2026


---
## 6 · All-Time vs Recent 2-Year Rankings

**What it shows:** Top/bottom 12 countries by volume-weighted avg tone — once for the full 2015–2026 period and once for the last 730 days only. Green = positive (above 0), red = negative.

**Story:** The most positive all-time countries (Vietnam, Kazakhstan, Estonia) remain outliers even recently. But the negative tail has deepened: Iran and Jamaica's negativity has intensified, and new entrants (Austria, Moldova, Ecuador) appear in the recent bottom-12. The shift is not just magnitude but also breadth — negativity is spreading to countries that were once indifferent.

**Process:** Volume-weighted avg tone per country (`sum(avg_tone × article_count) / sum(article_count)`), filtered to ≥100 articles (all-time) or ≥30 (recent).


In [8]:
cutoff = active.date.max() - pd.Timedelta(days=730)
cs_all = country_summary(active)
cs_rec = country_summary(active[active.date >= cutoff]).query('total_articles >= 20')

fig = make_subplots(rows=2,cols=2,
    subplot_titles=['All-time: Most Positive','Last 2 yrs: Most Positive',
                    'All-time: Most Negative','Last 2 yrs: Most Negative'],
    horizontal_spacing=0.12, vertical_spacing=0.12)
for col_i, cs_ in enumerate([cs_all, cs_rec], start=1):
    top = cs_.nlargest(15,'wavg_tone')
    bot = cs_.nsmallest(15,'wavg_tone')
    fig.add_trace(go.Bar(x=top.wavg_tone, y=top.name, orientation='h',
        marker_color=GREEN, showlegend=False,
        hovertemplate='%{y}: %{x:.2f}<extra></extra>'), row=1,col=col_i)
    fig.add_trace(go.Bar(x=bot.wavg_tone, y=bot.name[::-1], orientation='h',
        marker_color=RED, showlegend=False,
        hovertemplate='%{y}: %{x:.2f}<extra></extra>'), row=2,col=col_i)
fig.update_layout(**LAYOUT, height=580,
    title='Chart 6: All-time vs recent 2-year sentiment rankings')
fig.show()

---
## 7 · Who Switched Sides

**What it shows:** Slope chart connecting each country's avg tone in the *first half* of the dataset (2015–2020) to the *second half* (2021–2026). Red lines = turned more negative. Green lines = turned more positive.

**Story:** The chart is dominated by red. Almost every country that was neutral or mildly negative in 2015–2020 became more negative in 2021–2026. Only a handful moved positively — most notably Iran, which went from strongly negative to merely negative. The near-uniformity of direction across geographically and politically diverse nations underscores that this is a genuine sentiment shift, not a regional artefact.

**Process:** Country avg tone split at 2021-01-01. Countries need ≥30 articles in each period to be included. Lines coloured by sign of `(late − early)`.


In [9]:
mid = pd.Timestamp('2021-01-01')
cse = country_summary(active[active.date < mid]).query('total_articles >= 30')
csl = country_summary(active[active.date >= mid]).query('total_articles >= 30')
common = cse.index.intersection(csl.index)
delta = (csl.loc[common,'wavg_tone'] - cse.loc[common,'wavg_tone'])
top20 = delta.abs().nlargest(20).index
df7 = pd.DataFrame({'name':[COUNTRY_NAMES.get(c,c) for c in top20],
                    'early':cse.loc[top20,'wavg_tone'].values,
                    'late': csl.loc[top20,'wavg_tone'].values,
                    'delta':delta[top20].values}).sort_values('delta')

fig = go.Figure()
for _,row in df7.iterrows():
    col = RED if row.delta < 0 else GREEN
    fig.add_trace(go.Scatter(
        x=[0,1], y=[row.early, row.late], mode='lines+markers+text',
        line=dict(color=col,width=1.5),
        marker=dict(color=[GREEN if row.early>=0 else RED,
                            GREEN if row.late>=0 else RED], size=8),
        text=[row['name'],f"{row['late']:.1f}"],
        textposition=['middle left','middle right'],
        textfont=dict(size=9), showlegend=False,
        hovertemplate=f"{row['name']}<br>2015-20: {row.early:.2f} → 2021-26: {row.late:.2f}<extra></extra>"
    ))
fig.add_hline(y=0, line_width=1, line_color='#ccc', line_dash='dot')
fig.update_xaxes(tickvals=[0,1], ticktext=['2015–2020','2021–2026'], showgrid=False)
fig.update_yaxes(title='Avg tone')
fig.update_layout(**LAYOUT, height=580,
    title='Chart 7: Who switched sides — tone in first vs second half of decade')
fig.show()

---
## 8 · Geopolitical Blocs: G7 / BRICS / Anglosphere / Global South

**What it shows:** 30-day rolling avg tone for four geopolitical groupings, plotted on the same axes. If blocs diverged, we'd expect separation; if they track each other, the story is global.

**Story:** All four blocs move in near-perfect lockstep throughout the entire 11-year period. There is no meaningful difference between how G7 democracies and BRICS nations feel about Musk, nor between the Anglosphere and the Global South. This rules out political alignment, press-freedom level, or developmental status as drivers — the sentiment is driven by the *events themselves*, not by geopolitics.

**Process:** Countries assigned to blocs (non-exclusive for Global South). Daily avg tone weighted by article count per country, then averaged across bloc members. 30-day rolling mean applied.


In [10]:
blocs = {'G7':G7,'BRICS':BRICS,'Anglosphere':ANGLOSPHERE,'Global South':GLOBAL_SOUTH}
colors_b = {'G7':BLUE,'BRICS':RED,'Anglosphere':ORANGE,'Global South':GREEN}
fig = go.Figure()
for bloc,members in blocs.items():
    sub = active[active.country_iso3.isin(members)]
    if sub.empty: continue
    d8 = (sub.groupby('date')
            .apply(lambda g: np.average(g.avg_tone, weights=g.article_count))
            .reset_index(name='tone').sort_values('date'))
    d8['r30'] = d8.tone.rolling(30, center=True, min_periods=1).mean()
    fig.add_trace(go.Scatter(x=d8.date, y=d8.r30,
        line=dict(color=colors_b[bloc],width=2.2), name=bloc))
fig.add_hline(y=0, line_width=1, line_color='#ccc')
for d,lbl in EVENTS.items():
    fig.add_vline(x=pd.Timestamp(d).timestamp()*1000,
                  line_width=0.8, line_dash='dot', line_color='#bbb')
fig.update_layout(**LAYOUT, yaxis_title='30-day rolling avg tone',
    title='Chart 8: G7 vs BRICS vs Anglosphere vs Global South')
fig.show()

---
## 9 · Year-Over-Year Report Card (2015–2026)

**What it shows:** Three side-by-side bar charts: (1) global avg tone per year, (2) % of country-days with positive tone, (3) total articles per year (thousands).

**Story:** 2015 is the only year with a positive avg tone (+0.08). Every year since has been negative, steepening sharply after 2021. By 2025–2026 avg tone hit –1.65 to –1.76, with only 22–25% of country-days positive. Meanwhile, *volume* has exploded — 2025 saw 200k+ articles, the most ever. More coverage + worse tone = the feedback loop of polarised celebrity culture.

**Process:** Annual aggregation of `wavg_tone` (weighted) and binary positive-day counts. Article totals are raw sums across all countries.


In [11]:
active_yr = active.copy()
active_yr['year'] = active_yr.date.dt.year
yearly = (active_yr.groupby('year')
                   .apply(lambda g: pd.Series({
                       'wavg_tone': np.average(g.avg_tone, weights=g.article_count),
                       'total_articles': g.article_count.sum(),
                       'n_countries': g.country_iso3.nunique(),
                       'pct_positive': (g.avg_tone > 0).mean() * 100,
                   }))).reset_index()

fig = make_subplots(rows=1,cols=2,
    subplot_titles=['Global avg tone per year','% positive country-days per year'],
    horizontal_spacing=0.12)
fig.add_trace(go.Bar(x=yearly.year, y=yearly.wavg_tone,
    marker_color=[GREEN if t>=0 else RED for t in yearly.wavg_tone],
    hovertemplate='%{x}: %{y:.2f}<extra></extra>'), row=1,col=1)
fig.add_hline(y=0, line_color='#ccc', row=1,col=1)
fig.add_trace(go.Bar(x=yearly.year, y=yearly.pct_positive,
    marker_color=ORANGE, hovertemplate='%{x}: %{y:.1f}%<extra></extra>'), row=1,col=2)
fig.add_hline(y=50, line_dash='dot', line_color='#aaa', row=1,col=2)
fig.update_layout(**LAYOUT, height=400, showlegend=False,
    title='Chart 9: Year-over-year report card')
fig.show()
print(yearly[['year','wavg_tone','pct_positive','total_articles','n_countries']].to_string(index=False))

 year  wavg_tone  pct_positive  total_articles  n_countries
 2015   0.082580     55.005959          3540.0         68.0
 2016  -0.046642     53.233169         23276.0        111.0
 2017  -0.085701     48.943374         50602.0        119.0
 2018  -0.739009     37.181513         69349.0        124.0
 2019  -0.715263     41.870618         37138.0        107.0
 2020  -0.725551     40.511503         42545.0        110.0
 2021  -0.343156     43.323802         60309.0        124.0
 2022  -1.653765     21.003379        129457.0        121.0
 2023  -1.482586     23.928526        121628.0        125.0
 2024  -1.245271     29.491975        151152.0        130.0
 2025  -1.663596     22.022673        202358.0        136.0
 2026  -1.755509     25.144566         29994.0        109.0


---
## 10 · Quadrant Analysis: Positivity Ratio vs Negativity Intensity

**What it shows:** Each bubble is a country. X-axis = % of days with positive tone. Y-axis = mean tone on *negative* days only (i.e. how harsh negativity is). Bubble size = total article volume. Colour = overall avg tone.

**Story:** Most countries land in the lower-left quadrant — rarely positive, and harsh when negative. The **USA sits in the bottom-right** (low positivity ratio, very high negativity intensity, and the largest bubble) — loudest and most hostile. Vietnam and Kazakhstan are notable outliers, pushed right by their higher positivity ratios. No large-volume country sits in the upper half, confirming that scale of coverage correlates with negativity.

**Process:** Per-country metrics computed on rows with `article_count ≥ 5`. Quadrant lines drawn at dataset medians. Countries with ≥500 articles shown.


In [12]:
neg_int = (active[active.avg_tone < 0].groupby('country_iso3')['avg_tone']
           .mean().abs().rename('neg_intensity'))
q = cs.join(neg_int).dropna()
q['label'] = q.index.map(lambda x: COUNTRY_NAMES.get(x,x))

fig = go.Figure()
fig.add_vline(x=0.5, line_width=1, line_dash='dot', line_color='#ccc')
fig.add_hline(y=q.neg_intensity.median(), line_width=1, line_dash='dot', line_color='#ccc')
fig.add_trace(go.Scatter(
    x=q.pos_day_ratio, y=q.neg_intensity, mode='markers+text',
    marker=dict(
        size=np.sqrt(q.total_articles/q.total_articles.max())*45+5,
        color=q.wavg_tone,
        colorscale=[[0,RED],[0.5,'#f5f5f5'],[1,GREEN]],
        cmin=-4, cmax=4, showscale=True,
        colorbar=dict(title='Avg tone',thickness=12,len=0.6),
        line=dict(color='white',width=0.5),
    ),
    text=q.label, textposition='top center', textfont=dict(size=8),
    hovertemplate='<b>%{text}</b><br>Pos days: %{x:.0%}<br>Neg intensity: %{y:.2f}<extra></extra>',
))
fig.update_layout(**LAYOUT, height=540,
    title='Chart 10: Quadrant — positivity ratio vs negativity intensity (bubble = volume)')
fig.update_xaxes(title_text='Positive-day ratio', tickformat='.0%')
fig.update_yaxes(title_text='Neg intensity (abs mean tone on neg days)')
fig.show()

---
## 11 · Coverage Concentration

**What it shows:** Horizontal bar chart of top 15 countries by total article count, with percentage of global total labelled.

**Story:** The top 3 countries (USA 21%, UK 17%, Australia 10%) account for **48% of all articles**; the top 5 account for 61%. This is overwhelmingly an **English-language Anglophone media story**. The long tail of 140+ other countries contributes the remaining 39% — their coverage matters for geographic spread but not for driving the global average. This concentration means that shifts in US or UK editorial tone have an outsized effect on every aggregate metric in this notebook.

**Process:** `article_count.sum()` per country, sorted descending. Country ISO3 mapped to display names for readability.


In [13]:
tot = (active.groupby('country_iso3')['article_count'].sum()
       .sort_values(ascending=False).reset_index())
tot['name'] = tot.country_iso3.map(lambda x: COUNTRY_NAMES.get(x,x))
top40 = tot.head(40).copy()
rest = tot.iloc[40:]['article_count'].sum()
top40 = pd.concat([top40, pd.DataFrame([{
    'country_iso3':'OTH','name':'All others','article_count':rest
}])], ignore_index=True)

print(f"Top 1 country : {tot.iloc[0]['name']} — {tot.iloc[0]['article_count']:,} articles "
      f"({100*tot.iloc[0]['article_count']/tot['article_count'].sum():.1f}%)")
print(f"Top 3 countries: {100*tot.head(3)['article_count'].sum()/tot['article_count'].sum():.1f}% of all articles")
print(f"Top 5 countries: {100*tot.head(5)['article_count'].sum()/tot['article_count'].sum():.1f}% of all articles")

fig = go.Figure(go.Treemap(
    labels=top40.name, parents=['']*len(top40), values=top40.article_count,
    hovertemplate='<b>%{label}</b><br>%{value:,} articles (%{percentRoot:.1%})<extra></extra>',
    marker=dict(colors=top40.article_count,
                colorscale=[[0,'#f5ebe0'],[1,ORANGE]], showscale=False),
    textinfo='label+percent root', textfont=dict(size=11),
))
fig.update_layout(**LAYOUT, height=460, title='Chart 11: Coverage treemap — how concentrated is global attention?')
fig.show()

Top 1 country : United States — 195,212 articles (21.2%)
Top 3 countries: 47.8% of all articles
Top 5 countries: 61.3% of all articles


---
## 12 · Country × Year Sentiment Heatmap

**What it shows:** Grid of avg tone per country (rows) per year (columns), for the top 25 countries by article volume. Green = positive, red = negative, white/neutral = near-zero.

**Story:** The heatmap makes the temporal shift visceral. The left columns (2015–2017) are noticeably greener (or at least lighter red). From 2021 onward almost every cell turns deep red. India stands out — it remained comparatively neutral until 2022, then joined the global negative trend. The USA has been deep red the longest, reinforcing chart 11's finding that the Anglophone media set the tone early.

**Process:** Annual weighted avg tone per country, pivoted to country × year matrix. Countries sorted by all-time avg tone (most positive at top).


In [14]:
mid = pd.Timestamp('2020-01-01')
cse2 = country_summary(active[active.date < mid]).query('total_articles >= 20')
csl2 = country_summary(active[active.date >= mid]).query('total_articles >= 20')
common2 = cse2.index.intersection(csl2.index)
feat = pd.DataFrame({
    'tone':  cs.loc[cs.index.intersection(common2),'wavg_tone'],
    'std':   cs.loc[cs.index.intersection(common2),'tone_std'],
    'delta': csl2.loc[common2,'wavg_tone'] - cse2.loc[common2,'wavg_tone'],
}).dropna()

feat_w = whiten(feat.values)
_, labels = kmeans2(feat_w, 4, seed=42, minit='points')

cnames12 = {}
for lb in np.unique(labels):
    g = feat[labels==lb]
    t,d = g.tone.mean(), g.delta.mean()
    if t>0 and d>0:    cnames12[lb]='Positive & improving'
    elif t>0 and d<=0: cnames12[lb]='Positive but declining'
    elif t<=0 and d>0: cnames12[lb]='Negative but recovering'
    else:              cnames12[lb]='Negative & worsening'

map_df = pd.DataFrame({'iso3':feat.index,
    'cluster':[cnames12[l] for l in labels],
    'tone':feat.tone.values,
    'name':[COUNTRY_NAMES.get(c,c) for c in feat.index]})
cmap12 = {'Positive & improving':GREEN,'Positive but declining':'#86efac',
           'Negative but recovering':'#fca5a5','Negative & worsening':RED}

print("Cluster sizes:")
print(map_df.cluster.value_counts().to_string())

fig = px.choropleth(map_df, locations='iso3', locationmode='ISO-3',
    color='cluster', color_discrete_map=cmap12,
    hover_name='name', hover_data={'iso3':False,'tone':':.2f'})
fig.update_traces(marker_line_width=0.3, marker_line_color='white')
fig.update_layout(**LAYOUT, height=460,
    geo=dict(bgcolor='white',showframe=False,showcoastlines=True,
             coastlinecolor='#ddd',landcolor='#f0ede6',
             oceancolor='#e8f4f8',showocean=True),
    legend=dict(orientation='h',y=-0.05),
    title='Chart 12: Country clustering by sentiment profile')
fig.show()

Cluster sizes:
cluster
Negative & worsening       67
Negative but recovering    10


---
## 13 · Sentiment Momentum Map (last 180 days)

**What it shows:** For each country, the linear regression slope of `avg_tone` over the last 180 days. Positive slope (green) = sentiment improving. Negative slope (red) = sentiment worsening.

**Story:** As of mid-2026, **Bosnia, Kenya, and China are the top improvers** — possibly reflecting reduced scrutiny or more balanced recent coverage. Scandinavian countries (Norway, Finland) and Eastern Europe (Estonia, Croatia) are worsening fastest — nations that were already critical becoming more so. This chart is the 'forward-looking' signal: if these momentum patterns hold, the next 6 months will see Nordic and Eastern European media become as negative as the Anglosphere.

**Process:** OLS slope via `scipy.stats.linregress` on `(date_ordinal, avg_tone)` pairs per country. Countries need ≥20 data points in the 180-day window.


In [15]:
cutoff13 = active.date.max() - pd.Timedelta(days=180)
recent13 = active[active.date >= cutoff13]
slopes13 = {}
for iso, grp in recent13.groupby('country_iso3'):
    if len(grp) < 10: continue
    x = (grp.date - grp.date.min()).dt.days.values
    slope, *_ = stats.linregress(x, grp.avg_tone.values)
    slopes13[iso] = slope
sdf = pd.DataFrame({'iso3':list(slopes13.keys()), 'slope':list(slopes13.values()),
                    'name':[COUNTRY_NAMES.get(c,c) for c in slopes13]})

print(f"Top 5 improving: {sdf.nlargest(5,'slope')[['name','slope']].to_string(index=False)}")
print(f"Top 5 worsening: {sdf.nsmallest(5,'slope')[['name','slope']].to_string(index=False)}")

fig = px.choropleth(sdf, locations='iso3', locationmode='ISO-3',
    color='slope', hover_name='name',
    color_continuous_scale=[[0,RED],[0.5,'#f5f5f5'],[1,GREEN]],
    range_color=[-0.05,0.05], labels={'slope':'Tone slope/day'})
fig.update_traces(marker_line_width=0.3, marker_line_color='white')
fig.update_layout(**LAYOUT, height=440,
    geo=dict(bgcolor='white',showframe=False,showcoastlines=True,
             coastlinecolor='#ddd',landcolor='#f0ede6',
             oceancolor='#e8f4f8',showocean=True),
    title='Chart 13: Sentiment momentum — direction of travel (last 180 days)')
fig.show()

Top 5 improving:  name    slope
  BIH 0.048316
  KEN 0.032200
China 0.031458
  SVN 0.022356
Egypt 0.019779
Top 5 worsening: name     slope
 XKX -0.022971
 NAM -0.019120
 ECU -0.018266
 HRV -0.012526
 EST -0.010029


---
## 14 · Turning Point Grid

**What it shows:** For the top 30 countries by volume, a scatter dot placed at the *date of worst-ever daily tone* (x-axis) and coloured by the severity of that tone (colour scale).

**Story:** Almost all worst-ever days cluster between **2022 and 2025** — the Twitter acquisition, X rebrand, and DOGE period. Romania is an outlier, hitting its floor as early as 2017. Switzerland's worst day came in 2026, making it the *most recent* to reach its nadir. The progressive rightward drift of dots over time shows that each subsequent controversy has pushed more countries to new lows — the floor is not a stable resting point, it keeps falling.

**Process:** `argmin(avg_tone)` per country from the top-30 by volume. Dot size = article count on that day. Colour scale = tone at worst point.


In [16]:
top30 = cs.nlargest(30,'total_articles').index.tolist()
rows14 = []
for iso in top30:
    sub = active[active.country_iso3 == iso].sort_values('date')
    if sub.empty: continue
    wi = sub.avg_tone.idxmin()
    rows14.append({'name':COUNTRY_NAMES.get(iso,iso),
                   'worst_date':sub.loc[wi,'date'],
                   'worst_tone':sub.loc[wi,'avg_tone']})
tp = pd.DataFrame(rows14).sort_values('worst_date')

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=tp.worst_date, y=tp.name, mode='markers',
    marker=dict(
        size=(-tp.worst_tone/tp.worst_tone.abs().max()*22+6),
        color=tp.worst_tone,
        colorscale=[[0,RED],[0.5,'#f5d0c8'],[1,'#f5f5f5']],
        cmin=tp.worst_tone.min(), cmax=0, showscale=True,
        colorbar=dict(title='Tone',thickness=12),
        line=dict(color='white',width=0.5),
    ),
    hovertemplate='<b>%{y}</b><br>%{x|%b %d %Y}<br>Tone: %{marker.color:.2f}<extra></extra>',
))
for d,_ in EVENTS.items():
    fig.add_vline(x=pd.Timestamp(d).timestamp()*1000,
                  line_width=0.8, line_dash='dot', line_color='#bbb')
fig.update_layout(**LAYOUT, height=580,
    title="Chart 14: Turning point grid — each country's lowest day (size = severity)")
fig.update_xaxes(title_text='Date of worst-ever daily tone')
fig.update_yaxes(autorange='reversed')
fig.show()

---
## 15 · First Coverage + Honeymoon Gap

**What it shows:** Left panel — countries ranked by how many days elapsed between first *any* coverage and first *negative* coverage (the 'honeymoon period'). Right panel — date of first coverage (earlier = discovered Musk earlier).

**Story:** Belarus enjoyed a **2,000+ day honeymoon** — it first wrote about Musk positively and didn't produce a negative-avg-tone day for over five years. Most countries had honeymoon periods under 500 days. The USA, UK, and Australia (right panel, bottommost) all started covering Musk from the earliest dates in the dataset — and had almost no honeymoon, beginning critical coverage almost immediately. This suggests early adopter nations (those with established tech media) were more sceptical from the start, while later-discovering nations gave him more initial goodwill.

**Process:** Per country: `first_cov = min(date)`, `first_neg = min(date where avg_tone < 0)`. Honeymoon = `(first_neg − first_cov).days`. Countries needing a negative day but never having one are excluded.


In [17]:
rows15 = []
for iso, grp in active.groupby('country_iso3'):
    grp_s = grp.sort_values('date')
    first_cov = grp_s.date.min()
    neg = grp_s[grp_s.avg_tone < 0]
    first_neg = neg.date.min() if len(neg) > 0 else None
    gap = (first_neg - first_cov).days if first_neg else None
    rows15.append({'iso3':iso, 'name':COUNTRY_NAMES.get(iso,iso),
                   'first_cov':first_cov, 'gap':gap,
                   'total_articles':grp.article_count.sum()})

fd = (pd.DataFrame(rows15).dropna(subset=['gap'])
        .query('total_articles >= 50').sort_values('gap',ascending=False))

print(f"Longest honeymoons (days until first negative coverage):")
print(fd.head(10)[['name','gap','first_cov']].to_string(index=False))

fig = make_subplots(rows=1,cols=2,
    subplot_titles=['Honeymoon period (days to first negative report)',
                    'First coverage date'],
    horizontal_spacing=0.1)
top25 = fd.head(25)
fig.add_trace(go.Bar(x=top25.gap, y=top25.name, orientation='h',
    marker_color=ORANGE, hovertemplate='%{y}: %{x} days<extra></extra>'), row=1,col=1)
fd2 = fd.sort_values('first_cov').head(25)
fig.add_trace(go.Bar(x=fd2.first_cov, y=fd2.name, orientation='h',
    marker_color=BLUE, hovertemplate='%{y}: %{x|%b %Y}<extra></extra>'), row=1,col=2)
fig.update_layout(**LAYOUT, height=560, showlegend=False,
    title='Chart 15: First coverage + honeymoon gap per country')
fig.show()

Longest honeymoons (days until first negative coverage):
     name    gap  first_cov
      BLR 2122.0 2016-08-24
      TUN  860.0 2018-09-18
      NAM  839.0 2016-06-03
      CRI  661.0 2015-05-15
      DOM  519.0 2015-05-11
      MKD  495.0 2016-08-04
      XKX  443.0 2016-11-07
      BOL  432.0 2015-05-16
Singapore  430.0 2015-03-17
      PAN  405.0 2017-08-08


---
## Summary & Potential Storyline

---

### Running the website
```bash
python generate_eda_site.py   # → eda_elon_musk.html (self-contained, no CDN)
```

---

### 📖 Potential Storyline — *"The Fall of a Tech God"*

The data tells a coherent, three-act story of a celebrity who went from visionary to villain in the eyes of global media — and it happened in measurable steps:

---

**Act I — The Honeymoon (2015–2017): *The world discovers a real-life Tony Stark***

Coverage begins globally. 2015 is the only net-positive year on record (+0.08 avg tone, 55% positive days). Most countries write about SpaceX launches and Tesla milestones. The correlation between volume and tone is neutral-to-positive: big news days are good news days. Countries have long honeymoon periods (some 500–2,000+ days before a negative article). The CUSUM is flat — no structural pressure building yet.

---

**Act II — The Fractures (2018–2021): *Controversies accumulate, cracks appear***

The *'Pedo guy' tweet* (July 2018) is the first shock visible in every country's data. The SEC fraud charge follows weeks later. Average tone slips to –0.7. COVID-era behaviour (*'Tesla stock price is too high'*, lockdown defiance, Bitcoin manipulation accusations) triggers the **structural break in March 2020** — tone crosses –1.0 permanently. The K-means clustering shows countries splitting into two camps: those recalibrating downward, and a small bloc of outliers staying positive. G7 and BRICS track each other identically — this isn't political, it's personal.

---

**Act III — The Reckoning (2022–2026): *The world's most covered person, mostly negatively***

The **Twitter acquisition** (October 2022) is the watershed. The turning-point grid shows that most countries hit their all-time worst sentiment days between 2022 and 2025. Volume explodes to record highs (200k+ articles in 2025) while tone collapses to –1.66 to –1.76. The rolling volume–tone correlation flips to its most negative reading ever by 2026: the more he's covered, the worse it gets. The cumulative reputation index stands at 65 — a 36.5% drawdown from its 2016 peak with no stabilisation visible. Momentum maps show even previously sympathetic nations (Nordic countries, Eastern Europe) trending sharply negative.

---

### Key tension for the visualisation

> *More famous → more coverage → worse coverage.* 
> The world didn't lose interest in Elon Musk — it became more interested and more critical simultaneously. By 2026, global media publishes more articles about him per day than ever before, while delivering the harshest tone on record. Fame, for Musk, has become a reputational liability.

---

### Headline stats worth featuring
| Fact | Number |
|---|---|
| Only positive year | 2015 (+0.08) |
| Structural break | March 2020 |
| Peak reputation index | 102 (early 2016) |
| Current reputation index | 65 (–36.5% drawdown) |
| % positive days, 2015 | 55% |
| % positive days, 2026 | 25% |
| Top-3 countries' share of coverage | 48% (USA, UK, AUS) |
| Countries that improved 2015→2026 | < 5 out of 157 |
